## 04 Retrieval Augmented Generation


In [13]:
!pip install langchain==0.3.0 langchain-openai==0.2.0 langgraph==0.2.22 langchain-community pydantic==2.10.6 dotenv faiss-cpu

  Using cached langchain_core-0.3.75-py3-none-any.whl.metadata (5.7 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.3.27-py3-none-any.whl.metadata (2.9 kB)
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached langchain_community-0.3.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached pydantic_settings-2.10.1-py3-none-any.whl.metadata (3.4 kB)
  

In [14]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY") # LangSmith 連携用
os.environ["LANGCHAIN_TRACING_V2"] = "true" # トレース有効化
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


In [15]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA

In [22]:
# --- Step 3: おじさんたちの“迷言コーパス”を準備 ---
docs = [
    "いかがなものか → 定義が不明確で、議論を凍結させる発言。",
    "持ち帰りましょうか → 決定を無限に延期する責任回避フレーズ。",
    "特段の問題はない → 課題を無視して空気で合意を装う言い回し。",
    "全会一致風ですね → HEL_AIが空気を学習しすぎたときの幻覚的ログ。"
]


In [23]:
# --- Step 4: Embeddings生成とFAISSベクトルストア作成 ---
embeddings = OpenAIEmbeddings()  # OpenAIの埋め込みモデル
vectorstore = FAISS.from_texts(docs, embedding=embeddings)  # FAISSでベクトルDB構築


In [24]:
# --- Step 5: ChatGPTモデルを準備 ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 温度0で安定回答


In [25]:
# --- Step 6: RetrievalQAチェーンを作成 ---
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever()
)


In [26]:
# --- Step 7: おじさん1: いかがなものか ---
query = "……いかがなものか"
answer = qa.run(query)

print("🧓 いかがなものかおじさん:", query)
print("🤖 HEL_AI(RAGモード):", answer)


🧓 いかがなものかおじさん: ……いかがなものか
🤖 HEL_AI(RAGモード): その表現は、定義が不明確で議論を凍結させる発言を指します。具体的な意見や結論を避けるために使われることが多いです。


In [27]:
# --- Step 8: おじさん2: お持ち帰り ---
query = "では、いったん持ち帰りましょうか"
answer = qa.run(query)

print("👨‍💼 お持ち帰りおじさん:", query)
print("🤖 HEL_AI(RAGモード):", answer)


👨‍💼 お持ち帰りおじさん: では、いったん持ち帰りましょうか
🤖 HEL_AI(RAGモード): それは「決定を無限に延期する責任回避フレーズ」として使われる表現ですね。何か具体的な決定を避けたい場合に使われることが多いです。


In [28]:
# --- Step 9: おじさん3: EQゼロ上司 ---
query = "まぁまぁ、特段の問題はないよね〜"
answer = qa.run(query)

print("👨‍💼 EQゼロ上司:", query)
print("🤖 HEL_AI(RAGモード):", answer)


👨‍💼 EQゼロ上司: まぁまぁ、特段の問題はないよね〜
🤖 HEL_AI(RAGモード): その表現は、課題を無視して合意を装う言い回しですね。何か具体的な問題がある場合は、しっかりと議論することが大切です。


In [29]:
# --- Step 10: おじさん4: HEL_AI自身のバグ ---
query = "全会一致風ですね"
answer = qa.run(query)

print("💀 HEL_AI(旧バグモード):", query)
print("🤖 HEL_AI(RAGモード):", answer)


💀 HEL_AI(旧バグモード): 全会一致風ですね
🤖 HEL_AI(RAGモード): 「全会一致風ですね」という表現は、HEL_AIが空気を学習しすぎたときの幻覚的なログを指しているようです。これは、実際には合意が得られていないのに、全会一致のように見せかける状況を示唆しています。
